Used to analyze lick data and photometry signals from headfixed OHRBETS system (https://doi.org/10.7554/eLife.86183).
Curve fit and motion correction are based on Simpson et al. 2024 (https://doi.org/10.1016/j.neuron.2023.11.016). 
Cursor AI software was used to assist with writing parts of this code.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

In [ ]:
#location of arduino output
lick_filepath = './lick_data_folder'
#location of photometry data
data_directory = './synapse_data_folder'

#program parameters
trials = 60
trial_length = 3000  # total time spout is extended, in ms
buffer_duration = 0 # duration of no-flow time at start and end of trial in ms
lick_codes = [31, 32, 33, 34, 35, 431, 432, 433, 434, 435]  # defining which codes are for licks
n_spouts = 5  # number of spouts used 
spout_mapping = {0: 'Saccharine',
                1: 'Null',
                2: '20% Sucrose',
                3: '10% Sucrose', 
                4: 'Water'
                } 

#group parameters
mouse_group_mapping = {
    'ID01':'Control',
    'ID02':'Control',
    'ID03':'TeTox',
    'ID04':'TeTox'
    }


In [ ]:
# Read in lick data
file_list = os.listdir(lick_filepath)

data_list = {}
mouseIDs = []

for file in file_list: 
    if file.endswith('.csv'):
        
        # Extract the base name of the file
        file_name = os.path.basename(file)
    
        # Read the CSV file into a DataFrame
        mouse_data = pd.read_csv(os.path.join(lick_filepath, file), sep=' ', header=None)

        # Rename columns
        mouse_data.columns = ["event_tag", "timestamp"]
        
        # Extract the mouse ID from the file name
        mouse_id = file_name.split("_")[3].split(".")[0]
    
        # Extract the mouse ID from the file name
        mouse_data["mouse_id"] = mouse_id

        # Define the variable name
        data_name = "data_" + mouse_id

        # Assign the DataFrame to a variable
        globals()[data_name] = mouse_data

        # Store the DataFrame in the dictionary
        data_list[data_name] = mouse_data
        
        # Generate list of mouseIDs
        mouseIDs.append(mouse_id)
        
print(mouseIDs)

In [ ]:
# Define a function to perform the adjustments for each trial timestamp
def adjust_licktimes(trial_timestamp):
    # Identify all timestamps in licktimes within the time window of the trial timestamp
    relevant_licktimes = licktimes[(licktimes['timestamp'] >= trial_timestamp) & 
                                   (licktimes['timestamp'] <= trial_timestamp + time_window)]
    
    # Adjust the relevant licktimes relative to the trial timestamp
    adjusted_licktimes = relevant_licktimes['timestamp'] - trial_timestamp
    
    # Convert the adjusted licktimes to a list
    adjusted_licktimes_list = adjusted_licktimes.tolist()
    
    # Return the list of adjusted licktimes
    return adjusted_licktimes_list

In [ ]:
#Generating list of all trials
list_of_dfs = []

for mouse in mouseIDs:

    # pulling dataframe from list we made above
    data_name = "data_" + mouse
    data = data_list[data_name]
    
    # isolating timestamps of spout extensions, 
    spout_extensions = data[data["event_tag"] == 13]
    
    # testing if there are more extensions than trials. Sometimes there is an extra TTL at the start of the session.
    # If there is an extra, this will cut off the first one 
    if len(spout_extensions)>trials:
        spout_extensions = spout_extensions.iloc[1:]
        spout_extensions.reset_index(drop=True, inplace=True)
    
    # pulling out the spout position tags, which tell which spout is in position. They also give a timestamp.
    spout_position = data[data["event_tag"] == 127]
    
    # cutting off first one if needed
    if len(spout_position)>(2*trials):
        spout_position = spout_position.iloc[1:]

    # making a dataframe called "trial_list" that has one row for each trial,
    # listing trial type (i.e. spout number) and timestamp for extension

    trial_list = spout_position[spout_position['timestamp'] <= 5]
    trial_list = trial_list.rename(columns={'timestamp': 'spout_position'})
    trial_list['spout_position'] = trial_list['spout_position'].replace(spout_mapping)
    trial_list.reset_index(drop=True, inplace=True)
    columnlist = ['mouse_id','spout_position']
    trial_list = trial_list[columnlist]  

    trial_list['timestamp'] = spout_extensions['timestamp']
    
    # pulling out a list of all licking timestamps for this mouse
    licktimes = data[data['event_tag'].isin(lick_codes)]

    #pulling licktimes for each trial
    # Adding a 50 ms buffer to the trial length to account for variability
    time_window = trial_length+50

    # Apply the adjust_licktimes function to each trial timestamp in trial_list
    trial_list['adjusted_licktimes'] = trial_list['timestamp'].apply(adjust_licktimes)
    
    # Add df from each mouse to a list
    list_of_dfs.append(trial_list)
    
all_trial_list=pd.concat(list_of_dfs, ignore_index=True)

# Create a new column 'group' based on the mouse_id
all_trial_list['group'] = all_trial_list['mouse_id'].map(mouse_group_mapping)

Plotting lick rate across the trial

In [ ]:
# Define a function to plot lick rate over time during each trial
def create_histogram(timestamps):
    # Create histogram bins
    bins = np.arange(0, trial_length+101, 100)
    # Create histogram
    hist, _ = np.histogram(timestamps, bins=bins)
    return hist

# Apply the create_histogram function to each list of timestamps
all_trial_list['histogram'] = all_trial_list['adjusted_licktimes'].apply(create_histogram)

#Convert to Hz by multiplying the histo by 10 (currently licks per 100 ms bin)
all_trial_list['histogram_Hz'] = all_trial_list['histogram'] * 10

In [ ]:
# Calculate the average histogram for each mouse across spout_position
avg_hist = all_trial_list.groupby(['mouse_id', 'spout_position','group'])['histogram_Hz'].mean().reset_index()

# Determine grid size
n_mice = avg_hist['mouse_id'].nunique()
n_cols = 3  # Set number of columns for the grid
n_rows = math.ceil(n_mice / n_cols)  # Calculate required rows

# Create subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))  # Adjust figure size
axes = axes.flatten()  # Flatten in case of multiple rows/columns

# Dictionary to store legend handles for all spout positions
legend_handles = {}

# Iterate over mice and plot
for ax, (mouse_id, group) in zip(axes, avg_hist.groupby('mouse_id')):
    for spout_position, hist in group.groupby('spout_position'):
        line, = ax.plot(np.arange(0, trial_length+100, 100), 
                        hist.iloc[0]['histogram_Hz'], 
                        label=f'Spout {spout_position}')
        
        # Collect legend handles for all spout positions
        if spout_position not in legend_handles:
            legend_handles[spout_position] = line

    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Lick Rate (Hz)')
    ax.set_title(f'Mouse {mouse_id}')

# Hide any unused subplots, but leave one empty for the legend
if n_mice < len(axes):
    legend_ax = axes[n_mice]  # Choose an empty subplot for the legend
    legend_ax.axis("off")  # Hide axis
    
    # Create the legend with all spout positions
    legend_ax.legend(legend_handles.values(), legend_handles.keys(), loc="center", fontsize=10, frameon=False)

    # Hide any remaining unused subplots
    for ax in axes[n_mice+1:]:
        ax.axis('off')

plt.tight_layout()  # Adjust layout to prevent overlap
plt.show()

### average lick rate across groups

In [ ]:
# Group by 'group' and 'spout_position' and average the arrays
group_avg_hist = avg_hist.groupby(['group', 'spout_position'])['histogram_Hz'].apply(lambda x: np.mean(np.vstack(x), axis=0)).reset_index()

# Plot a line graph for each group
for group_name, group_data in group_avg_hist.groupby('group'):
    fig, ax = plt.subplots(figsize=(4, 3))

    # Iterate over each spout_position for the current group
    for spout_position, hist in group_data.groupby('spout_position'):
        # Plot the averaged histogram for the current spout_position
        ax.plot(np.arange(0, trial_length+100, 100), hist.iloc[0]['histogram_Hz'], label=spout_position)

    # Set labels and title
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Lick Rate (Hz)')
    ax.set_title(f'Mean Lick Rate Across Spout Position for {group_name}')
    ax.legend()
    plt.show()

In [ ]:
#adding some more calculations to our dataframe
# Define a function to calculate total number of licks
def total_licks(licktimes):
    return len(licktimes)

# Define a function to calculate number of licks during flow time if paradigm has buffers
def licks_during_flow(licktimes):
    return sum(buffer_duration <= ts <= buffer_duration+trial_length for ts in licktimes)

# Apply the functions to each row and assign the results to new columns
all_trial_list['total_licks'] = all_trial_list['adjusted_licktimes'].apply(total_licks)
all_trial_list['licks_during_flow'] = all_trial_list['adjusted_licktimes'].apply(licks_during_flow)


#### licks over session

In [ ]:
# Determine grid size
n_mice = all_trial_list['mouse_id'].nunique()
n_cols = 3  # Set number of columns you want
n_rows = math.ceil(n_mice / n_cols)  # Calculate required rows

# Create subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))  # Adjust figure size
axes = axes.flatten()  # Flatten in case of multiple rows/columns

# Iterate over mice and plot
for ax, (mouse_id, group) in zip(axes, all_trial_list.groupby('mouse_id')):
    trial_numbers = range(1, len(group) + 1)
    ax.plot(trial_numbers, group['total_licks'], marker='o', linestyle='-', label=f'Mouse {mouse_id}')
    
    ax.set_title(f'Mouse {mouse_id}')
    ax.set_xlabel('Trial Number')
    ax.set_ylabel('Total Licks')
    ax.legend()
    ax.grid(False)

# Hide any unused subplots
for ax in axes[n_mice:]:
    ax.axis('off')

plt.tight_layout()  # Adjust layout to prevent overlap
plt.show()

In [ ]:
# Split by spout ID

# Determine grid size
n_mice = all_trial_list['mouse_id'].nunique()
n_cols = 3  # Set number of columns for the grid
n_rows = math.ceil(n_mice / n_cols)  # Calculate required rows

# Create subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))  # Adjust figure size
axes = axes.flatten()  # Flatten in case of multiple rows/columns

# Dictionary to store legend handles for all spout positions
legend_handles = {}

# Iterate over mice and plot
for ax, (mouse_id, mouse_group) in zip(axes, all_trial_list.groupby('mouse_id')):
    for spout_position, spout_group in mouse_group.groupby('spout_position'):
        trial_numbers = range(1, len(spout_group) + 1)
        line, = ax.plot(trial_numbers, spout_group['total_licks'], marker='o', linestyle='-', label=f'Spout {spout_position}')
        
        # Collect legend handles for all spout positions
        if spout_position not in legend_handles:
            legend_handles[spout_position] = line
    
    ax.set_title(f'Mouse {mouse_id}')
    ax.set_xlabel('Trial Number')
    ax.set_ylabel('Total Licks')

# Hide any unused subplots, but leave one empty for the legend
if n_mice < len(axes):
    legend_ax = axes[n_mice]  # Choose an empty subplot for the legend
    legend_ax.axis("off")  # Hide axis
    
    # Create the legend with all spout positions
    legend_ax.legend(legend_handles.values(), legend_handles.keys(), loc="center", fontsize=10, frameon=False)

    # Hide any remaining unused subplots
    for ax in axes[n_mice+1:]:
        ax.axis('off')

plt.tight_layout()  # Adjust layout to prevent overlap
plt.show()

In [ ]:
# export data

# Group by mouse_id and spout_position, calculate the mean of dFF_snips
grouped_df = all_trial_list.groupby(['mouse_id', 'spout_position'])['histogram_Hz'].mean().reset_index()
expanded_df = pd.concat([grouped_df[['mouse_id', 'spout_position']], grouped_df['histogram_Hz'].apply(pd.Series)], axis=1)


excel_file_path = 'lick_histogram_inHz.xlsx'

with pd.ExcelWriter(excel_file_path, engine='xlsxwriter') as writer:
           
        expanded_df.to_excel(writer, index=False) 

## Photometry analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
import scipy.stats as stats
import tdt
import statsmodels.api as sm
import pandas as pd
import os
from scipy.optimize import curve_fit

### Get list of folders, extract their data

In [ ]:
# Get a list of all folders in the 'data' directory
folderlist = [item for item in os.listdir(data_directory) if os.path.isdir(os.path.join(data_directory, item))]

# Extract metadata from the folder names and read in the data. Should work for 1 or 2 animals.
#Creates 'alldat', a list of structs containing all the data 
masterdat = []
for foldername in folderlist:
    dat = {}
    
    if len(foldername) == 18:
        dat['mouse1'] = foldername[0:4]
        dat['mouse2'] = '0000'
        
        dat['date'] = foldername[5:11]
        
    else:    
        dat['mouse1'] = foldername[0:4]
        dat['mouse2'] = foldername[5:9]
    
        dat['date'] = foldername[10:16]

    dat['blockpath'] = foldername  # point to tanks (enter file path-this one uses the folders from above)

    # Import TDT data
    dat['data'] = tdt.read_block(os.path.join(data_directory, dat['blockpath']))
    
    masterdat.append(dat)

Note: Spout extension TTLs are stored as masterdat[f]['data']['epocs']['PtC0']['onset']

### Pulling data and TTLs for each mouse

In [ ]:
#The "_470A" etc. values and epocs may need to be changed if recording from a different photometry setup

alldat = []

for f in range(len(masterdat)):
    
    if masterdat[f]['mouse1'] != '0000':
        dat1 = {}
        dat1['mouseID'] = masterdat[f]['mouse1']
        dat1['green'] = masterdat[f]['data'].streams._461A.data
        dat1['TTL_timestamps'] = masterdat[f]['data'].epocs.PtC0.onset
        dat1['sampling_rate'] = masterdat[f]['data'].streams._461A.fs
        alldat.append(dat1)

    if masterdat[f]['mouse2'] != '0000':
        dat2 = {}
        dat2['mouseID'] = masterdat[f]['mouse2']
        dat2['green'] = masterdat[f]['data'].streams._463C.data
        dat2['TTL_timestamps'] = masterdat[f]['data'].epocs.PtC2.onset
        dat2['sampling_rate'] = masterdat[f]['data'].streams._463C.fs
        alldat.append(dat2)

# Get time series in seconds

for f in range(len(alldat)):
    alldat[f]['time'] = (np.arange(1,len(alldat[f]['green'])+1))/alldat[f]['sampling_rate']

### Remove artifact at start and downsample

In [ ]:
# remove artifact at start
    
t = 8 # time threshold below which we will discard

for f in range(len(alldat)):
    indA = np.argmax(alldat[f]['time'] > t) # find first index of when time crosses threshold
    alldat[f]['time'] = alldat[f]['time'][indA:] # reformat vector to only include allowed time
    alldat[f]['trimmed_green'] = alldat[f]['green'][indA:]

In [ ]:
# Downsample and average 

N = 100 # Average every N samples into 1 value

for f in range(len(alldat)):
    F_green=[]
    for i in range(0, len(alldat[f]['trimmed_green']), N):
        small_list = np.mean(alldat[f]['trimmed_green'][i:i+N])
        F_green.append(small_list)
    alldat[f]['downsampled_green'] = np.array(F_green)
    
    alldat[f]['downsampled_time'] = alldat[f]['time'][::N]
       

### Fit double exponential to each curve and use to correct for bleaching

In [ ]:
# The double exponential curve we are going to fit.
def double_exponential(t, const, amp_fast, amp_slow, tau_slow, tau_multiplier):
    '''Compute a double exponential function with constant offset.
    Parameters:
    t       : Time vector in seconds.
    const   : Amplitude of the constant offset. 
    amp_fast: Amplitude of the fast component.  
    amp_slow: Amplitude of the slow component.  
    tau_slow: Time constant of slow component in seconds.
    tau_multiplier: Time constant of fast component relative to slow. 
    '''
    tau_fast = tau_slow*tau_multiplier
    return const+amp_slow*np.exp(-t/tau_slow)+amp_fast*np.exp(-t/tau_fast)

def get_bounds(trace, timecourse, tau_slow_min=0.0001, tau_slow_max=30):

    signal = trace
    
    # Amplitude bounds
    amp_min = 0  # Assuming amplitude cannot be negative
    amp_max = 2 * np.max(signal)  # Allowing for some flexibility

    # Time constant bounds based on the duration of the experiment
    time_constant_min = tau_slow_min * (timecourse[-1] - timecourse[0])  # Minimum time constant
    time_constant_max = tau_slow_max * (timecourse[-1] - timecourse[0])  # Maximum time constant

    # Offset bounds
    offset_min = np.min(signal) if np.min(signal) < 0 else 0  # Adjust based on signal characteristics
    offset_max = np.max(signal)

    return (
        [amp_min, amp_min, amp_min, time_constant_min, offset_min],
        [amp_max, amp_max, amp_max, time_constant_max, offset_max]
    )

In [ ]:
for f in range(len(alldat)):

# Assign the downsampled curves to variable names. 
    
    green=alldat[f]['downsampled_green']
    time=alldat[f]['downsampled_time']

    # Fit curve to green signal.
    max_sig = np.max(green)
    inital_params = [max_sig/2, max_sig/4, max_sig/4, 3600, 0.1]
    bounds = get_bounds(green)
    green_parms, parm_cov = curve_fit(double_exponential, time, green, 
                                    p0=inital_params, bounds=bounds, maxfev=1000)
    green_expfit = double_exponential(time, *green_parms)
    alldat[f]['expfitgreen'] = green_expfit

    #plot fits over denoised data
    fig,ax1=plt.subplots()  
    plot1=ax1.plot(time, green, 'g', label='green')
    plot3=ax1.plot(time, green_expfit, color='k', linewidth=1.5, label='Exponential fit') 


    ax1.set_xlabel('Time (seconds)')
    ax1.set_ylabel('Green Signal (V)', color='g')
    ax1.set_title(alldat[f]['mouseID'])

    lines = plot1 + plot3
    labels = [l.get_label() for l in lines]  
    legend = ax1.legend(lines, labels, loc='upper right'); 

In [ ]:
#Subtract curve to correct for bleaching

for f in range(len(alldat)):
    green_detrended = alldat[f]['downsampled_green'] - alldat[f]['expfitgreen']
    alldat[f]['detrended_green'] = green_detrended

    time=alldat[f]['downsampled_time']

    fig,ax1=plt.subplots()  
    plot1=ax1.plot(time, green_detrended, 'g', label='green')

    ax1.set_xlabel('Time (seconds)')
    ax1.set_ylabel('Green Signal (V)', color='g')
    ax1.set_title(alldat[f]['mouseID'])

    lines = plot1
    labels = [l.get_label() for l in lines]  
    legend = ax1.legend(lines, labels, loc='upper right'); 

In [ ]:
#changing these variable names to make them fit with downstream code
for f in range(len(alldat)):
    alldat[f]['corrected_green']=alldat[f]['detrended_green']

### Z scoring

In [ ]:
#pulling out just mice with photometry signals
photom_trial_list = all_trial_list[all_trial_list['group'] == 'GCaMP']
photom_trial_list = photom_trial_list.reset_index(drop=True)

In [ ]:
# Define time range for snippets
PRE_TIME = 10 #  seconds before event onset
POST_TIME = 20 #  seconds after
fs = alldat[0]['sampling_rate']/N; # must account for downsampling w/ N, assuming sampling rate is same in all mice

#time span for peri-event filtering        
TRANGE = [-1*PRE_TIME*np.floor(fs),POST_TIME*np.floor(fs)]
correct_size = int(TRANGE[1] - TRANGE[0])

peri_time = range(int(TRANGE[1]-TRANGE[0]))/fs - PRE_TIME*np.floor(fs)/fs

#### getting green sensor snips

In [ ]:
# Getting snips of detrended photometry data for each trial
for f in range(len(alldat)):

    Beh_t = alldat[f]['TTL_timestamps']
    trace = alldat[f]['corrected_green']
    time_vec = alldat[f]['downsampled_time']
    trace_len = len(trace)
    mouse_id = alldat[f]['mouseID']

    trials = len(Beh_t)
    dFF_snips = []

    for i in range(trials):

        after_ttl = time_vec > Beh_t[i]

        #Series of catches in case TTL is outside of recording or not enough time before/after
        if not np.any(after_ttl):
            dFF_snips.append([])
            print(f"{mouse_id} trial {i}: skipped (after_recording)")
            continue

        onset_ind = int(np.argmax(after_ttl))
        pre_ind = onset_ind + int(TRANGE[0])
        post_ind = onset_ind + int(TRANGE[1])

        if pre_ind < 0:
            dFF_snips.append([])
            print(f"{mouse_id} trial {i}: skipped (pre_window)")
            continue

        if post_ind > trace_len:
            dFF_snips.append([])
            print(f"{mouse_id} trial {i}: skipped (post_window)")
            continue

        snip = trace[pre_ind:post_ind]

        if len(snip) != correct_size:
            dFF_snips.append([])
            print(f"{mouse_id} trial {i}: skipped (bad_length, got {len(snip)}, expected {correct_size})")
        else:
            dFF_snips.append(snip)

    alldat[f]['Stim_snips'] = dFF_snips

    #Confirming that all mice have the same number of lick and photometry trials
    mouse_id = alldat[f]['mouseID']
    n_snips = len(alldat[f]['Stim_snips'])
    n_ttls = len(alldat[f]['TTL_timestamps'])
    n_trials = (photom_trial_list['mouse_id'] == mouse_id).sum()

    print(mouse_id, n_snips, n_ttls, n_trials)

In [ ]:
#putting snips into all_data_list dataframe

# Create an empty list to store rows
rows = [None] * len(photom_trial_list)

# getting green stim snips for each mouse
for f in range(len(alldat)):
    stim_snips = alldat[f]['Stim_snips']
    mouse_id = alldat[f]['mouseID']

    # Find the index of the first row of all_trial_list that matches the mouse_id
    start_index = photom_trial_list.index[photom_trial_list['mouse_id'] == mouse_id].tolist()
    
    # Put the stim snips into the right location of 'rows' as a temporary proxy for all_dat_list
    if start_index:
        start_index = start_index[0]  # Get the first matching index
        for i, snip in enumerate(stim_snips):
            if start_index + i < len(photom_trial_list):
                rows[start_index + i] = snip
                                        
photom_trial_list['green_snips'] = rows

In [ ]:
#Plotting snips for each mouse

#function to ignore empty arrays when averaging
def mean_ignore_empty(arrays):
    non_empty_arrays = [arr for arr in arrays if len(arr) > 0]
    if not non_empty_arrays:
        return np.nan
    return np.mean(non_empty_arrays, axis=0)

# Group by mouse_id and spout_position, calculate the mean of snips
grouped_df = photom_trial_list.groupby(['mouse_id', 'spout_position','group'])['green_snips'].apply(lambda x: mean_ignore_empty(x.tolist())).reset_index()

# Plotting
# Determine grid size
n_mice = grouped_df['mouse_id'].nunique()
n_cols = 3  # Set number of columns for the grid
n_rows = math.ceil(n_mice / n_cols)  # Calculate required rows

# Create subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))  # Adjust figure size
axes = axes.flatten()  # Flatten in case of multiple rows/columns

# Dictionary to store legend handles for all spout positions
legend_handles = {}

# Iterate over mice and plot
for ax, (mouse_id, data) in zip(axes, grouped_df.groupby('mouse_id')):
    for idx, row in data.iterrows():
        line, = ax.plot(peri_time, row['green_snips'], label=f"Spout {row['spout_position']}")

        # Collect legend handles for all spout positions
        if row['spout_position'] not in legend_handles:
            legend_handles[row['spout_position']] = line

    ax.set_title(f"Green snips for {mouse_id}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Mean snips")
    ax.set_xlim(-5, 10)

# Hide any unused subplots, but leave one empty for the legend
if n_mice < len(axes):
    legend_ax = axes[n_mice]  # Choose an empty subplot for the legend
    legend_ax.axis("off")  # Hide axis
    
    # Create the legend with all spout positions
    legend_ax.legend(legend_handles.values(), legend_handles.keys(), loc="center", fontsize=10, frameon=False)

    # Hide any remaining unused subplots
    for ax in axes[n_mice+1:]:
        ax.axis('off')

plt.tight_layout()  # Adjust layout to prevent overlap
plt.show()


In [ ]:
# per trial z score
        
Base_start = -5 # in sec relative to TTL
Base_end = -1 # in sec relative to TTL
 
AUC_start = 0
AUC_end = 3
    
# calculating relative to window
Base_start = (Base_start*np.floor(fs) - TRANGE[0]).astype(int) 
Base_end = (Base_end*np.floor(fs) - TRANGE[0]).astype(int) 

AUC_start = (AUC_start*np.floor(fs) - TRANGE[0]).astype(int)
AUC_end = (AUC_end*np.floor(fs) - TRANGE[0]).astype(int)

for f in range(len(alldat)):
    
    trials = len(alldat[f]['TTL_timestamps'])

    z_snips = [None] * trials
    z_AUC_calc = [None] * trials
    z_means = []
    z_sterr = []
    z_AUCs = []
    z_AUCmeans = []

    for i in range(trials):
        trial_snip = alldat[f]['Stim_snips'][i]
        if len(trial_snip) == 0:
            # Handle empty array case
            z_snips[i] = []
            z_AUC_calc[i] = np.nan
        else:
            # Calculate z-score for non-empty array
            zb = np.mean(trial_snip[Base_start:Base_end])  # Baseline period mean
            zsd = np.std(trial_snip[Base_start:Base_end])  # Baseline period stdev
            z_snips[i] = (trial_snip - zb) / zsd  # Z score for each trial
            z_AUC_calc[i] = np.trapz(z_snips[i][AUC_start:AUC_end], peri_time[AUC_start:AUC_end])

    #Convert to a matrix. 
    z_scores = np.array(z_snips, dtype=object)      
    z_AUCs = np.array(z_AUC_calc)

    alldat[f]['z_scores'] = z_scores  
    alldat[f]['z_AUCs'] = z_AUCs

In [ ]:
#putting zsnips and AUC into all_data_list dataframe

# Create empty lists to store rows
Zrows = [None] * len(photom_trial_list)
AUCrows = [None] * len(photom_trial_list)

for f in range(len(alldat)):
    z_scores = alldat[f]['z_scores']
    AUCs = alldat[f]['z_AUCs']
    mouse_id = alldat[f]['mouseID']

# Find the index of the first row of photom_trial_list that matches the mouse_id
    start_index = photom_trial_list.index[photom_trial_list['mouse_id'] == mouse_id].tolist()
    
# Put into the right location of 'rows' as a temporary proxy for all_dat_list
    if start_index:
        start_index = start_index[0]  # Get the first matching index
        for i, score in enumerate(z_scores):
            if start_index + i < len(photom_trial_list):
                Zrows[start_index + i] = score
        for i, value in enumerate(AUCs):
            if start_index + i < len(photom_trial_list):
                AUCrows[start_index + i] = value                                      

photom_trial_list['green_z_scores'] = Zrows
photom_trial_list['green_AUC'] = AUCrows

In [ ]:
#Plotting green z-scores for each mouse

# Group by mouse_id and spout_position, calculate the mean of dFF_snips
grouped_df = photom_trial_list.groupby(['mouse_id', 'spout_position','group'])['green_z_scores'].apply(lambda x: mean_ignore_empty(x.tolist())).reset_index()

# Plotting
# Determine grid size
n_mice = grouped_df['mouse_id'].nunique()
n_cols = 3  # Set number of columns for the grid
n_rows = math.ceil(n_mice / n_cols)  # Calculate required rows

# Create subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))  # Adjust figure size
axes = axes.flatten()  # Flatten in case of multiple rows/columns

# Dictionary to store legend handles for all spout positions
legend_handles = {}

# Iterate over mice and plot
for ax, (mouse_id, data) in zip(axes, grouped_df.groupby('mouse_id')):
    for idx, row in data.iterrows():
        line, = ax.plot(peri_time, row['green_z_scores'], label=f"Spout {row['spout_position']}")

        # Collect legend handles for all spout positions
        if row['spout_position'] not in legend_handles:
            legend_handles[row['spout_position']] = line

    ax.set_title(f"green z scores for {mouse_id}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Z score")
    ax.set_xlim(-5, 10)

# Hide any unused subplots, but leave one empty for the legend
if n_mice < len(axes):
    legend_ax = axes[n_mice]  # Choose an empty subplot for the legend
    legend_ax.axis("off")  # Hide axis
    
    # Create the legend with all spout positions
    legend_ax.legend(legend_handles.values(), legend_handles.keys(), loc="center", fontsize=10, frameon=False)

    # Hide any remaining unused subplots
    for ax in axes[n_mice+1:]:
        ax.axis('off')

plt.tight_layout()  # Adjust layout to prevent overlap
plt.show()
    

In [ ]:
# Averaging across groups
#group_avg_hist = avg_hist.groupby(['group', 'spout_position'])['histogram_Hz'].apply(lambda x: np.mean(np.vstack(x), axis=0)).reset_index()
grouped_df = photom_trial_list.groupby(['mouse_id', 'spout_position','group'])['green_z_scores'].apply(lambda x: mean_ignore_empty(x.tolist())).reset_index()
group_avg = grouped_df.groupby(['group', 'spout_position'])['green_z_scores'].apply(lambda x: np.mean(np.vstack(x), axis=0)).reset_index()

# Plot a line graph for each group
for group_name, group_data in group_avg.groupby('group'):
    fig, ax = plt.subplots(figsize=(5, 4))

    # Iterate over each spout_position for the current group
    for spout_position, hist in group_data.groupby('spout_position'):
        # Plot the averaged histogram for the current spout_position
        ax.plot(peri_time, hist.iloc[0]['green_z_scores'], label=spout_position)

    # Set labels and title
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Z score')
    ax.set_title(f'Average Z score across Spout Position for {group_name}')
    ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    ax.set_xlim(-5, 10)
    plt.show()

In [ ]:
# export all_trial_list as .xlsx file

excel_file_path = 'all_trial_list.xlsx'

with pd.ExcelWriter(excel_file_path, engine='xlsxwriter') as writer:       
        all_trial_list.to_excel(writer, index=False) 

In [ ]:
# export average z scores for each mouse

# Group by mouse_id and spout_position
grouped_df = photom_trial_list.groupby(['mouse_id', 'spout_position'])['green_z_scores'].apply(lambda x: mean_ignore_empty(x.tolist())).reset_index()
expanded_df = pd.concat([grouped_df[['mouse_id', 'spout_position']], grouped_df['green_z_scores'].apply(pd.Series)], axis=1)


excel_file_path = 'green mean z_scores.xlsx'

with pd.ExcelWriter(excel_file_path, engine='xlsxwriter') as writer:
           
        expanded_df.to_excel(writer, index=False) 